# Hybrid NER Pipeline — Full Method Coverage

All NER methods developed for the DACKAR RCA workflow, tested end-to-end.

| # | Method | Module | Output |
|---|--------|--------|--------|
| 1 | **Regex: Equipment IDs** | `equipment_ID_extractor` | Tag strings: `P-101A`, `MOV-204` |
| 2 | **Regex: Document Refs** | `doc_ref_extractor` | DocRef tuples: `CR-2024-0451`, `WO-87342` |
| 3 | **Regex: Alarm IDs** | `alarm_id_extractor` | AlarmRef tuples: `ALM-101`, `ANN-A-15` |
| 4 | **spaCy: Measurements** | `spacy_annotator` | `{value, unit, entity_type, text}` |
| 5 | **spaCy: Temporal** | `spacy_annotator` | Refs, relations, qualifiers, lag_hours |
| 6 | **spaCy: Locations & Conjectures** | `spacy_annotator` | Spatial terms, hedge markers |
| 7 | **NLP: Causal Relations** | `causal_condition_adapter` | Cause→effect pairs, condition state |
| 8 | **Hybrid NER: Entities** | `ner_adapter` + `hybrid_ner/` | Systems, components, mechanisms, outcomes, actions |
| 9 | **End-to-End: NERSeed** | `ner_adapter.ner_seed_provider_from_pipeline` | Unified seed across all methods |

Pipeline stages for Hybrid NER (item 8)
-----------------------------------------
1. **Gazetteer** — exact-phrase matching against `tag_keywords_lists.xlsx`
2. **AnchoredNPGenerator** — noun phrases anchored to gazetteer tokens; `emit_min_tokens=False`
3. **DescriptionEmbedGenerator** — SBERT cosine similarity; `score_threshold=0.65`
4. **LLMDisambiguator** — uncertain confidence band `[0.40, 0.65]`
5. **CompatibilityEngine** — schema-rule conflict resolution
6. **Postprocessor** — nested-span pruning

In [1]:
import sys, os, re, json
import pandas as pd

# ── sys.path setup ─────────────────────────────────────────────────────────────
# Notebook lives at: .../DACKAR/src/dackar/RCA/tests/test_NER_1/
# SRC_DIR  : .../DACKAR/src          (4 levels up — lets Python find 'dackar' package)
# NER_PKG_DIR: .../DACKAR/src/dackar/RCA/ner  (where data/ and source modules live)
NOTEBOOK_DIR = os.path.abspath(os.getcwd())
SRC_DIR      = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..', '..', '..', '..'))
NER_PKG_DIR  = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..', '..', 'ner'))
for p in [SRC_DIR, NER_PKG_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Core pipeline
from dackar.RCA.ner.hybrid_ner.schema import SchemaLoader
from dackar.RCA.ner.hybrid_ner.models import Document
from dackar.RCA.ner.hybrid_ner.llm_disambiguator import LLMConfig
from dackar.RCA.ner.ner_adapter import build_ner_pipeline, ner_seed_provider_from_pipeline

# ID / reference extractors
from dackar.RCA.ner.equipment_ID_extractor import extract_equipment_ids
from dackar.RCA.ner.doc_ref_extractor import extract_doc_refs, load_doc_ref_profile
from dackar.RCA.ner.alarm_id_extractor import extract_alarm_refs

# spaCy annotator
from dackar.RCA.ner.spacy_annotator import build_spacy_annotator

# Causal extractor (requires spaCy dep-parser + dackar.causal)
try:
    from dackar.RCA.ner.causal_condition_adapter import extract_stage5_causal_condition
    CAUSAL_AVAILABLE = True
except Exception as _causal_err:
    CAUSAL_AVAILABLE = False
    print(f"[WARN] causal_condition_adapter not available: {_causal_err}")

print("Imports OK")
print(f"  SRC_DIR     : {SRC_DIR}")
print(f"  NER_PKG_DIR : {NER_PKG_DIR}")
print(f"  Causal NLP  : {'available' if CAUSAL_AVAILABLE else 'unavailable'}")

Imports OK
  SRC_DIR     : /Users/mandd/projects/DACKAR/src
  NER_PKG_DIR : /Users/mandd/projects/DACKAR/src/dackar/RCA/ner
  Causal NLP  : available


In [2]:
SCHEMA_PATH = os.path.join(NER_PKG_DIR, "data", "group-schema.json")
GAZ_PATH    = os.path.join(NER_PKG_DIR, "data", "tag_keywords_lists.xlsx")

# ── Validate schema / gazetteer consistency ────────────────────────────────────
schema = SchemaLoader.load(SCHEMA_PATH)
print(f"Schema labels : {len(schema.label_to_group)}")
print(f"Schema groups : {len(schema.group_to_labels)}")
print(f"Groups        : {sorted(schema.group_to_labels.keys())}")

schema_raw  = json.load(open(SCHEMA_PATH))
schema_lbls = set(schema_raw.keys())
xl          = pd.ExcelFile(GAZ_PATH)
excel_lbls  = set()
for sh in xl.sheet_names:
    df = xl.parse(sh)
    for col in df.columns:
        m = re.search(r"\[([^\]]+)\]\s*$", str(col))
        if m:
            excel_lbls.add(m.group(1).strip())

missing = sorted(excel_lbls - schema_lbls)
extra   = sorted(schema_lbls - excel_lbls)
print(f"\nExcel labels missing in schema : {missing}  (total {len(missing)})")
print(f"Schema labels missing in excel : {extra}  (total {len(extra)})")
print(f"\nWorking directory : {os.getcwd()}")
print(f"SRC_DIR     : {SRC_DIR}")
print(f"NER_PKG_DIR : {NER_PKG_DIR}")

Schema labels : 39
Schema groups : 10
Groups        : ['G10_ENV_BIO_CONTEXT', 'G1_PHYSICAL_COMPONENT', 'G2_PHYSICAL_ASSET_SYSTEM', 'G3_MATERIAL_CHEMISTRY', 'G4_MECHANISM_PROCESS', 'G5_FAILURE_OUTCOME', 'G6_ACTION_MAINT_SURV', 'G7_TOOL_METHOD', 'G8_PROPERTY_STATE', 'G9_DIAGNOSTIC_QUAL']

Excel labels missing in schema : []  (total 0)
Schema labels missing in excel : []  (total 0)

Working directory : /Users/mandd/projects/DACKAR/src/dackar/RCA/tests/test_NER_1
SRC_DIR     : /Users/mandd/projects/DACKAR/src
NER_PKG_DIR : /Users/mandd/projects/DACKAR/src/dackar/RCA/ner


In [3]:
# ── LLM health check ───────────────────────────────────────────────────────────
import requests

LLM_URL   = "http://localhost:11434/v1/chat/completions"
LLM_MODEL = "llama3.2:latest"

try:
    r = requests.post(
        LLM_URL,
        json={"model": LLM_MODEL, "messages": [{"role": "user", "content": "Reply with exactly OK."}], "temperature": 0},
        timeout=10,
    )
    r.raise_for_status()
    reply = r.json()["choices"][0]["message"]["content"].strip()
    print(f"LLM health : OK  (model={LLM_MODEL}, reply={reply!r})")
    LLM_AVAILABLE = True
except Exception as e:
    print(f"LLM health : UNAVAILABLE ({e})")
    print("Pipeline will run without LLM disambiguation.")
    LLM_AVAILABLE = False

LLM health : OK  (model=llama3.2:latest, reply='OK.')


---
## Section 1 — Regex: Equipment ID Extractor

Pattern-based extraction of nuclear plant equipment / tag identifiers.
Handles common conventions: `P-101A`, `MOV-204`, `PT-1102`, `AFW-P-101`, etc.

False-positive filters remove year-like numbers, date ranges, and document-reference prefixes (CR, WO, SOP).

In [4]:
# ── Equipment ID extractor ─────────────────────────────────────────────────────
#
# Tests:
#   Standard tags        : P-101A, MOV-204, PT-1102
#   Multi-segment tags   : AFW-P-101, RHR-MOV-204A
#   Should NOT extract   : CR-2024 (doc ref prefix), 2024-01 (date), A-1 (too short)

texts_equip = {
    "standard_tags": (
        "Work order for pump P-101A and valve MOV-204. "
        "Pressure transmitter PT-1102 indicated high pressure. "
        "Flow control valve FCV-100 was manually isolated."
    ),
    "multi_segment": (
        "The AFW-P-101 pump was found degraded. "
        "RHR-MOV-204A failed to open on demand. "
        "Reference CR-2024-0451 documents the event (not a tag)."
    ),
    "fp_cases": (
        "Event occurred between 2024-01 and 2024-03. "
        "Section A-1 describes the methodology. "
        "Temperature range was 10-15 degrees."
    ),
}

for label, text in texts_equip.items():
    ids = extract_equipment_ids(text)
    print(f"[{label}]")
    print(f"  Text : {text[:100]}...")
    print(f"  Tags : {ids}")
    print()

[standard_tags]
  Text : Work order for pump P-101A and valve MOV-204. Pressure transmitter PT-1102 indicated high pressure. ...
  Tags : ['P-101A', 'MOV-204', 'PT-1102', 'FCV-100']

[multi_segment]
  Text : The AFW-P-101 pump was found degraded. RHR-MOV-204A failed to open on demand. Reference CR-2024-0451...
  Tags : ['AFW-P-101', 'RHR-MOV-204A']

[fp_cases]
  Text : Event occurred between 2024-01 and 2024-03. Section A-1 describes the methodology. Temperature range...
  Tags : []



---
## Section 2 — Profile-based: Document Reference Extractor

Extracts cross-reference IDs for nuclear document types (CR, WO, ECA, SOP, LER, GL, etc.)
driven by `data/plant_profiles/default_plant_profile.json`.

Each match returns a `DocRef(doc_type, label, raw, norm)` namedtuple.

In [5]:
# ── Document reference extractor ───────────────────────────────────────────────

PLANT_PROFILE_PATH = os.path.join(NER_PKG_DIR, "data", "plant_profiles", "default_plant_profile.json")
plant_profile = load_doc_ref_profile(PLANT_PROFILE_PATH)
print(f"Plant profile loaded. Doc-ref types: "
      f"{[e['type'] for e in plant_profile.get('doc_ref_types', [])]}\n")

texts_docref = {
    "cr_and_wo": (
        "As documented in CR-2024-0451 and follow-up WO-87342, the pump bearing "
        "was found degraded. See also WO 87401 for corrective actions."
    ),
    "procedures_and_regulatory": (
        "Operators followed SOP-OP-4.2 and EOP-E0 during the event. "
        "LER 2023-001-00 was submitted per 10 CFR 50.73. "
        "GL 2004-02 commitments were verified against this condition."
    ),
    "multi_type": (
        "NCR-2024-112 was initiated. CAP-45678 references PER-2024-330. "
        "ECA-1.2 was entered when SI signal was received. "
        "IN 2023-05 (NRC Information Notice) is applicable."
    ),
}

for label, text in texts_docref.items():
    refs = extract_doc_refs(text, profile=plant_profile)
    print(f"[{label}]")
    for r in refs:
        print(f"  {r.doc_type:6s}  label={r.label:18s}  raw={r.raw!r:25s}  norm={r.norm}")
    print()

Plant profile loaded. Doc-ref types: ['CR', 'CAP', 'WO', 'PM', 'ECA', 'SOP', 'LER', 'GL', 'IN', 'BUL', 'OE', 'NCR']

[cr_and_wo]
  CR      label=doc_ref_cr          raw='CR-2024-0451'             norm=CR-2024-0451
  WO      label=doc_ref_wo          raw='WO-87342'                 norm=WO-87342
  WO      label=doc_ref_wo          raw='WO 87401'                 norm=WO-87401

[procedures_and_regulatory]
  SOP     label=doc_ref_sop         raw='SOP-OP-4.2'               norm=SOP-OP-4.2
  GL      label=doc_ref_gl          raw='GL 2004-02'               norm=GL-2004-02

[multi_type]
  CAP     label=doc_ref_cap         raw='CAP-45678'                norm=CAP-45678
  IN      label=doc_ref_in          raw='IN 2023-05'               norm=IN-2023-05
  NCR     label=doc_ref_ncr         raw='NCR-2024-112'             norm=NCR-2024-112
  NCR     label=doc_ref_ncr         raw='PER-2024-330'             norm=PER-2024-330



---
## Section 3 — Profile-based: Alarm ID Extractor

Extracts alarm and annunciator IDs (ALM, ANN, process tag alarms, SCRAM/SI conditions).
Each match returns `AlarmRef(pattern_name, raw, norm, score)` where `score` is the
pattern confidence from the plant profile.

In [6]:
# ── Alarm ID extractor ─────────────────────────────────────────────────────────

texts_alarm = {
    "standard_alarms": (
        "Annunciator ALM-101 alarmed in the control room. "
        "ANN-A-15 (High Reactor Coolant Pressure) was received at 03:42. "
        "Alarm ALM 204 cleared after operator action."
    ),
    "process_tag_alarms": (
        "PT-1102 high-high alarm was received. "
        "LT-0045 low level alarm actuated the ECCS. "
        "TT-301 temperature alarm was acknowledged by the operator."
    ),
    "setpoint_conditions": (
        "SI signal was initiated on P-4 (Reactor Coolant Low Pressure). "
        "AFAS actuation followed LOSP signal. "
        "SCRAM was initiated by T/M-2 (High Neutron Flux)."
    ),
}

for label, text in texts_alarm.items():
    refs = extract_alarm_refs(text, profile=plant_profile)
    print(f"[{label}]")
    if refs:
        for r in refs:
            print(f"  pattern={r.pattern_name:20s}  raw={r.raw!r:20s}  norm={r.norm:20s}  score={r.score:.2f}")
    else:
        print("  (no alarms detected)")
    print()

[standard_alarms]
  pattern=short_alarm_id        raw='ALM-101'             norm=ALM-101               score=0.88
  pattern=annunciator_window    raw='ANN-A-15'            norm=ANN-A-15              score=0.85

[process_tag_alarms]
  pattern=kla_point             raw='PT-1102'             norm=PT-1102               score=0.65
  pattern=kla_point             raw='LT-0045'             norm=LT-0045               score=0.65

[setpoint_conditions]
  (no alarms detected)



---
## Section 4 — spaCy Annotator

Six plant-specific spaCy pipeline components (loaded once, shared across Tier 1 indexing
and Tier 2 evidence scoring):

| Component | Output field | Example |
|-----------|-------------|---------|
| `unit_entity` | `measurements` | `{value:250.0, unit:"gpm", entity_type:"flow"}` |
| `Temporal` | `temporal_refs` | `"48 hours"`, `"March 14 2025"` |
| `temporal_relation_entity` | `temporal_relations` | `{text:"before", sub_label:"temporal_relation_reverse_order"}` |
| `temporal_attribute_entity` | `temporal_qualifiers` | `"approximately"`, `"roughly"` |
| `location_entity` | `locations` | `{text:"downstream", sub_label:"location_down"}` |
| `conjecture_entity` | `conjectures` | `"possibly"`, `"suspected"` |

`lag_hours` is derived from the first parseable duration in `temporal_refs`.
`lag_is_approximate` is True when `temporal_qualifiers` are present alongside `lag_hours`.

In [7]:
# ── Build spaCy annotator (one-time, shared across Tier 1 and Tier 2) ─────────
annotator = build_spacy_annotator(nlp_model="en_core_web_sm")
print("spaCy annotator ready:", annotator.nlp.pipe_names)

spaCy annotator ready: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner', 'Temporal', 'temporal_relation_entity', 'temporal_attribute_entity', 'location_entity', 'conjecture_entity', 'unit_entity']


In [8]:
# ── spaCy: measurements and process variables ──────────────────────────────────

def show_spacy(text, label=""):
    r = annotator.annotate(text)
    print(f"[{label}]  {text[:90]}{'...' if len(text)>90 else ''}")
    if r.measurements:
        print(f"  measurements     : {r.measurements}")
    if r.temporal_refs:
        print(f"  temporal_refs    : {r.temporal_refs}")
        print(f"  lag_hours        : {r.lag_hours}  approximate={r.lag_is_approximate}")
    if r.temporal_relations:
        print(f"  temporal_rels    : {r.temporal_relations}")
    if r.temporal_qualifiers:
        print(f"  temporal_qual    : {r.temporal_qualifiers}")
    if r.locations:
        print(f"  locations        : {r.locations}")
    if r.conjectures:
        print(f"  conjectures      : {r.conjectures}")
        print(f"  conjecture_frac  : {r.conjecture_fraction():.2f}")
    dominant = r.dominant_temporal_relation()
    if dominant:
        print(f"  dominant_rel     : {dominant}")
    print()
    return r

# Test 1: process variable measurements
r_meas = show_spacy(
    "Reactor coolant flow was 250 gpm. Temperature reached 285°F. "
    "Containment pressure was 14.7 psia and decreased to 10.2 psia over 72 hours.",
    label="measurements"
)

# Test 2: temporal references and ordering
r_temp = show_spacy(
    "Approximately 48 hours before the pump trip, vibration levels began rising. "
    "The seal failure was observed 2 days later on March 14 2025.",
    label="temporal"
)

# Test 3: locations and conjectures
r_loc = show_spacy(
    "Possible erosion was found downstream of the orifice plate. "
    "Suspected cavitation damage was likely caused by low suction pressure upstream of the pump.",
    label="locations+conjectures"
)

[measurements]  Reactor coolant flow was 250 gpm. Temperature reached 285°F. Containment pressure was 14.7...
  measurements     : [{'value': 250.0, 'unit': 'gram picometre', 'entity_type': 'unknown', 'text': '250 gpm'}, {'value': 285.0, 'unit': 'degree fahrenheit', 'entity_type': 'temperature', 'text': '285°F'}, {'value': 14.7, 'unit': 'pound per square inch year', 'entity_type': 'dynamic viscosity', 'text': '14.7 psia'}, {'value': 10.2, 'unit': 'pound per square inch year', 'entity_type': 'dynamic viscosity', 'text': '10.2 psia'}]
  temporal_refs    : ['72 hours']
  lag_hours        : 72.0  approximate=False
  locations        : [{'text': 'over', 'sub_label': 'location_up'}]

[temporal]  Approximately 48 hours before the pump trip, vibration levels began rising. The seal failu...
  temporal_refs    : ['Approximately 48 hours', 'before', '2 days later', 'on March 14 2025']
  lag_hours        : 48.0  approximate=False

[locations+conjectures]  Possible erosion was found downstream of t

### Known spaCy Annotator Limitations

| Issue | Detail |
|-------|--------|
| Unit resolution | Abbreviations like ,  may resolve to verbose SI forms (, ). Add site-specific unit aliases to fix. |
| Location coverage | Directional terms (, ) are **not** detected by the current  patterns. |
| Location false positives | Short words like  may match  patterns. |
| Temporal false positives | Doc-ref IDs like  can appear in . Use the  output to post-filter. |
| LLM disambiguation | Small general-purpose models (e.g. ) lack nuclear-domain knowledge; domain-fine-tuned or larger models strongly preferred. |


---
## Section 5 — NLP: Causal Relations (Stage 5)

`extract_stage5_causal_condition()` runs a three-tier extraction cascade:

1. **CausalSentence** (primary) — dep-tree traversal for explicit connectors
   (`caused by`, `due to`, `resulted in`, `led to`, …)
2. **CausalSimple** (fallback) — simpler dep-tree, fewer constraints
3. **LLM tier** (optional) — implicit causal detection when rule-based tiers return nothing

Output fields:
- `extracted_causal_statements` — list of `{cause_text, connector, effect_text, confidence, source}`
- `condition_state.as_found` / `as_left` — `"acceptable" | "degraded" | "failed" | "unknown"`
- `summary_flags` — boolean flags for quick filtering

In [9]:
# ── Causal relation extractor ──────────────────────────────────────────────────

def show_causal(text, doc_id="DOC", chunk_index=0, doc_type="CR", section_role="work_performed"):
    if not CAUSAL_AVAILABLE:
        print("Causal extractor not available — skipping.")
        return None

    result = extract_stage5_causal_condition(
        doc_id=doc_id,
        chunk_index=chunk_index,
        chunk_text=text,
        doc_type=doc_type,
        section_role=section_role,
        nlp=annotator.nlp,
        llm_cfg={"enabled": LLM_AVAILABLE, "http_url": LLM_URL, "model": LLM_MODEL,
                 "temperature": 0.0, "timeout": 15} if LLM_AVAILABLE else None,
    )

    print(f"[{doc_id}]  status={result['status']}  extractor_used={result['extractor']['used']}")
    flags = result["summary_flags"]
    print(f"  flags : causal={flags['has_explicit_causal_statement']}  "
          f"condition={flags['has_condition_state']}  "
          f"conjecture={flags['has_conjecture']}  "
          f"negation={flags['has_negation']}")
    cs = result.get("condition_state", {})
    print(f"  condition_state : as_found={cs.get('as_found')}  as_left={cs.get('as_left')}")
    stmts = result.get("extracted_causal_statements", [])
    print(f"  causal statements ({len(stmts)}):")
    for s in stmts[:5]:
        print(f"    [{s.get('source','?'):15s}] conf={s.get('confidence',0):.2f}  "
              f"cause={s.get('cause_text','')!r}  →  effect={s.get('effect_text','')!r}")
    if result.get("errors"):
        print(f"  errors : {result['errors']}")
    print()
    return result

# Test 1: explicit connectors — expect causal=True, ≥2 statements
r_causal_1 = show_causal(
    "Bearing wear caused excessive vibration, resulting in seal failure and coolant leakage. "
    "The degraded seal was found as-found in failed condition. "
    "After replacement the system was returned to as-left acceptable condition.",
    doc_id="DOC_EXPLICIT_CAUSAL",
    section_role="work_performed",
)

# Test 2: prepositional connector — expect causal=True, ≥1 statement
r_causal_2 = show_causal(
    "Inspection found the valve in a degraded condition due to actuator spring fatigue. "
    "No causal connector sentence here, but as-found state is clearly degraded.",
    doc_id="DOC_CONDITION_STATE",
    section_role="as_found",
)

# Test 3: negation and conjecture — expect conjecture=True, negation=True
r_causal_3 = show_causal(
    "It is possible that erosion caused the thinning, but this has not been confirmed. "
    "The failure did not result from improper installation.",
    doc_id="DOC_HEDGE",
    section_role="analysis",
)

[DOC_EXPLICIT_CAUSAL]  status=ok  extractor_used=CausalSentence
  flags : causal=True  condition=True  conjecture=False  negation=False
  condition_state : as_found=failed  as_left=acceptable
  causal statements (2):
    [dep_fallback+llm_repair] conf=0.95  cause='bearing wear'  →  effect='excessive vibration'
    [dep_fallback+llm_repair] conf=0.95  cause='bearing wear'  →  effect='seal failure'



[DOC_CONDITION_STATE]  status=ok  extractor_used=CausalSentence
  flags : causal=True  condition=True  conjecture=False  negation=True
  condition_state : as_found=degraded  as_left=None
  causal statements (1):
    [dep_fallback   ] conf=0.95  cause='actuator spring fatigue'  →  effect='Inspection found the valve in a degraded condition'



[DOC_HEDGE]  status=ok  extractor_used=CausalSentence
  flags : causal=True  condition=True  conjecture=True  negation=True
  condition_state : as_found=unknown  as_left=None
  causal statements (2):
    [dep_fallback   ] conf=0.90  cause='erosion'  →  effect='the thinning'
    [dep_fallback   ] conf=0.85  cause='The failure'  →  effect='improper installation'



---
## Section 6 — Hybrid NER Pipeline

Gazetteer + AnchoredNPGenerator + DescriptionEmbedGenerator + LLMDisambiguator.
Classifies spans into: systems, components, mechanisms, outcomes, maintenance/surveillance actions, tools, properties.

In [10]:
# ── Build pipeline via the canonical adapter ───────────────────────────────────
#
# Key settings:
#   generator_mode="anchored_np"
#     - Single AnchoredNPGenerator with unified token index (all gazetteer labels)
#     - emit_min_tokens=False  →  no sub-token _min_ spans
#   np_score_threshold=0.65
#     - DescriptionEmbedGenerator requires cosine ≥ 0.65 for anchored NP candidates
#       (replaces the old 0.30 that let weak system-group labels through)
#
# LLM confidence-band settings:
#   uncertain_score_floor=0.40    → below this, span is too weak even for LLM
#   uncertain_score_ceiling=0.65  → above this, embedding result is confident enough
#   high_conf_bypass_score=0.85   → gazetteer hits ≥ 0.85 bypass LLM entirely

llm_cfg = {
    "enabled"                : LLM_AVAILABLE,
    "http_url"               : LLM_URL,
    "model"                  : LLM_MODEL,
    "temperature"            : 0.0,
    "timeout"                : 10,
    "uncertain_score_floor"  : 0.40,
    "uncertain_score_ceiling": 0.65,
    "high_conf_bypass_score" : 0.85,
}

pipe = build_ner_pipeline(
    schema_json_path   = SCHEMA_PATH,
    gazetteer_xl       = GAZ_PATH,
    label_json         = SCHEMA_PATH,
    llm_cfg            = llm_cfg,
    generator_mode     = "anchored_np",
    np_score_threshold = 0.65,
)

print("Pipeline built.")
print(f"  Generators : {[type(g).__name__ for g in pipe.generators]}")
print(f"  desc_gen   : threshold={pipe.desc_gen.score_threshold}")
print(f"  LLM active : {pipe.llm_disambiguator is not None and getattr(pipe.llm_disambiguator, 'llm_ok', False)}")

Pipeline built.
  Generators : ['GazetteerGenerator', 'AnchoredNPGenerator']
  desc_gen   : threshold=0.65
  LLM active : True


In [11]:
# ── Display helper ─────────────────────────────────────────────────────────────

def run_and_display(pipe, text, doc_id="TEST", show_decisions=True):
    doc = Document(doc_id=doc_id, text=text)
    res = pipe.run(doc)

    print(f"\n{'─'*70}")
    print(f"[{doc_id}] {text!r}")
    print(f"{'─'*70}")

    print("\nEntities:")
    if res.entities:
        for e in res.entities:
            print(f"  [{e.start:3d}:{e.end:3d}]  {e.text!r:35s}  labels={e.labels}  groups={e.groups}")
    else:
        print("  (none)")

    if show_decisions:
        print("\nDecisions:")
        for d in res.decisions:
            spans_out = [(s.text, s.labels) for s in d.output_spans]
            notes = "; ".join(d.notes) if d.notes else ""
            print(f"  {d.action:8s}  {spans_out}" + (f"  [{notes}]" if notes else ""))

    print(f"\n  → {len(res.entities)} entities, {len(res.decisions)} decisions")
    return res

In [12]:
# ── Test 1 : original notebook text (regression test) ─────────────────────────
#
# EXPECTED (correct) entities:
#   'adhesive wear'   → deg_mech   (G4_MECHANISM_PROCESS)
#   'bearing'         → comp_mech_rot (G1_PHYSICAL_COMPONENT)
#   'acid attack'     → deg_mech   (G4_MECHANISM_PROCESS)
#   'gasket'          → comp_mech_spec (G1_PHYSICAL_COMPONENT)
#   'seal'            → comp_mech_struct (G1) OR 'seal degradation' → deg_mech (G4)
#   'pitting damage'  → deg_mech   (G4_MECHANISM_PROCESS)   ← was MISSED before
#   'cam shaft'       → comp_mech_rot (G1_PHYSICAL_COMPONENT)
#
# SHOULD NOT APPEAR (ghost entities fixed by emit_min_tokens=False):
#   'adhesive', 'wear', 'acid', 'attack', 'cam', 'shaft', 'degradation'

text1 = (
    "Work order: Inspection found adhesive wear on the bearing and evidence of acid attack on the gasket. "
    "Additional notes mention seal degradation and pitting damage near the cam shaft region."
)
res1 = run_and_display(pipe, text1, doc_id="DOC_ORIGINAL")


──────────────────────────────────────────────────────────────────────
[DOC_ORIGINAL] 'Work order: Inspection found adhesive wear on the bearing and evidence of acid attack on the gasket. Additional notes mention seal degradation and pitting damage near the cam shaft region.'
──────────────────────────────────────────────────────────────────────

Entities:
  [ 29: 42]  'adhesive wear'                      labels=['deg_mech']  groups=['G4_MECHANISM_PROCESS']
  [ 46: 57]  'the bearing'                        labels=['ast_eln']  groups=['G2_PHYSICAL_ASSET_SYSTEM']
  [ 50: 57]  'bearing'                            labels=['comp_mech_rot']  groups=['G1_PHYSICAL_COMPONENT']
  [ 74: 85]  'acid attack'                        labels=['deg_mech']  groups=['G4_MECHANISM_PROCESS']
  [ 79: 85]  'attack'                             labels=['event']  groups=['G5_FAILURE_OUTCOME']
  [ 89: 99]  'the gasket'                         labels=['ast_I_C']  groups=['G2_PHYSICAL_ASSET_SYSTEM']
  [ 93: 99]  'g

In [13]:
# ── Test 2 : OOV degradation phrases ──────────────────────────────────────────
#
# These compound phrases are unlikely to appear verbatim in the gazetteer.
# AnchoredNPGenerator picks them up via anchor tokens (fretting, corrosion,
# oxide, fatigue); DescriptionEmbedGenerator (SBERT) classifies them.
#
# EXPECTED:
#   'fretting corrosion'  → deg_mech (G4)
#   'pump impeller'       → comp_fluid or comp_mech_rot (G1)
#   'oxide layer buildup' → deg_mech or prop (G4 / G8)
#   'bearing fatigue'     → deg_mech (G4)

text2 = (
    "Investigation identified fretting corrosion on the pump impeller surface. "
    "Oxide layer buildup was observed, consistent with bearing fatigue failure mode."
)
res2 = run_and_display(pipe, text2, doc_id="DOC_OOV")


──────────────────────────────────────────────────────────────────────
[DOC_OOV] 'Investigation identified fretting corrosion on the pump impeller surface. Oxide layer buildup was observed, consistent with bearing fatigue failure mode.'
──────────────────────────────────────────────────────────────────────

Entities:
  [  0: 13]  'Investigation'                      labels=['surv_tool']  groups=['G7_TOOL_METHOD']
  [ 25: 43]  'fretting corrosion'                 labels=['deg_mech']  groups=['G4_MECHANISM_PROCESS']
  [ 51: 55]  'pump'                               labels=['ast_hyd_pne']  groups=['G2_PHYSICAL_ASSET_SYSTEM']
  [ 56: 64]  'impeller'                           labels=['comp_mech_spec']  groups=['G1_PHYSICAL_COMPONENT']
  [ 56: 64]  'impeller'                           labels=['ast_eln']  groups=['G2_PHYSICAL_ASSET_SYSTEM']
  [ 65: 72]  'surface'                            labels=['ast_eln']  groups=['G2_PHYSICAL_ASSET_SYSTEM']
  [ 74: 79]  'Oxide'                           

In [14]:
# ── Test 3 : failure events and maintenance actions ────────────────────────────
#
# EXPECTED:
#   'valve stem leakage' → deg_mech or fail_type_n (G4 / G5)
#   'motor bearing'      → comp_mech_rot (G1)
#   'visual inspection'  → surv_ops_n (G6)
#   'replaced'           → mnt_ops (G6)
#   'flow transmitter'   → comp_inst (G1) or similar
#   'instrument drift'   → deg_mech or prop (G4 / G8)

text3 = (
    "Technician performed visual inspection and found valve stem leakage. "
    "Motor bearing was replaced during corrective maintenance. "
    "Flow transmitter showed instrument drift above acceptance criteria."
)
res3 = run_and_display(pipe, text3, doc_id="DOC_EVENTS")


──────────────────────────────────────────────────────────────────────
[DOC_EVENTS] 'Technician performed visual inspection and found valve stem leakage. Motor bearing was replaced during corrective maintenance. Flow transmitter showed instrument drift above acceptance criteria.'
──────────────────────────────────────────────────────────────────────

Entities:
  [ 21: 38]  'visual inspection'                  labels=['ast_eln']  groups=['G2_PHYSICAL_ASSET_SYSTEM']
  [ 49: 54]  'valve'                              labels=['comp_hyd_pne']  groups=['G1_PHYSICAL_COMPONENT']
  [ 49: 54]  'valve'                              labels=['ast_hyd_pne']  groups=['G2_PHYSICAL_ASSET_SYSTEM']
  [ 60: 67]  'leakage'                            labels=['fail_type_n']  groups=['G5_FAILURE_OUTCOME']
  [ 60: 67]  'leakage'                            labels=['ast_hyd_pne']  groups=['G2_PHYSICAL_ASSET_SYSTEM']
  [ 69: 82]  'Motor bearing'                      labels=['comp_mech_rot']  groups=['G1_PHYSICAL_C

In [15]:
# ── Test 4 : LLM-band ambiguous spans ─────────────────────────────────────────
#
# These spans should land in the uncertain confidence band [0.40, 0.65] so the
# LLMDisambiguator is triggered (when LLM_AVAILABLE=True).
#
# EXPECTED:
#   'stress corrosion cracking'  → deg_mech (G4)
#   'reactor coolant pump'       → syst or comp (G2 / G1)
#   'vibration signature'        → prop or surv_ops_n (G8 / G6)

text4 = (
    "Analysis attributed the failure to stress corrosion cracking initiated at the "
    "reactor coolant pump inlet nozzle. Vibration signature monitoring confirmed progressive damage."
)
res4 = run_and_display(pipe, text4, doc_id="DOC_AMBIGUOUS")


──────────────────────────────────────────────────────────────────────
[DOC_AMBIGUOUS] 'Analysis attributed the failure to stress corrosion cracking initiated at the reactor coolant pump inlet nozzle. Vibration signature monitoring confirmed progressive damage.'
──────────────────────────────────────────────────────────────────────

Entities:
  [ 20: 31]  'the failure'                        labels=['ast_mech']  groups=['G2_PHYSICAL_ASSET_SYSTEM']
  [ 24: 31]  'failure'                            labels=['fail_type_n']  groups=['G5_FAILURE_OUTCOME']
  [ 35: 41]  'stress'                             labels=['prop']  groups=['G8_PROPERTY_STATE']
  [ 35: 51]  'stress corrosion'                   labels=['deg_mech']  groups=['G4_MECHANISM_PROCESS']
  [ 35: 60]  'stress corrosion cracking'          labels=['ast_I_C']  groups=['G2_PHYSICAL_ASSET_SYSTEM']
  [ 74:111]  'the reactor coolant pump inlet nozzle'  labels=['ast_eln']  groups=['G2_PHYSICAL_ASSET_SYSTEM']
  [ 86: 98]  'coolant pump' 

In [16]:
# ── LLM disambiguation audit ───────────────────────────────────────────────────
#
# Reports every span the LLMDisambiguator was consulted for.
# should_call() rules (in priority order):
#   Rule 1 : gazetteer hit score ≥ 0.85          → bypass LLM (trust gazetteer)
#   Rule 2 : no proposed labels at all            → call LLM
#   Rule 3 : all labels lack a known schema group → call LLM
#   Rule 4 : multiple competing schema groups     → call LLM
#   Rule 5 : best score in [0.40, 0.65]           → call LLM
#   Rule 6 : best score < 0.40                    → skip (too weak)
#   Rule 7 : best score > 0.65                    → skip (confident enough)

if pipe.llm_disambiguator is not None:
    cache = getattr(pipe.llm_disambiguator, '_cache', {})
    print(f"LLM cache entries (unique spans sent to LLM): {len(cache)}")
    for key, entry in list(cache.items())[:30]:
        resp = entry.get('resp', {})
        print(f"  span_key : {key}")
        print(f"  label    : {resp.get('label')}  score={resp.get('score')}")
        print(f"  rationale: {resp.get('rationale')}")
        print()
else:
    print("LLM disambiguator not active.")

LLM cache entries (unique spans sent to LLM): 16
  span_key : DOC_ORIGINAL:46:57:the bearing
  label    : ast_eln  score=0.8
  rationale: The bearing is likely made of a material that has been affected by the acid attack.

  span_key : DOC_ORIGINAL:89:99:the gasket
  label    : ast_I_C  score=0.8
  rationale: Adhesive wear on the bearing and acid attack on the gasket indicate a chemical issue.

  span_key : DOC_ORIGINAL:131:142:degradation
  label    : deg_mech  score=0.8
  rationale: degradation is a common term in mechanical failure analysis

  span_key : DOC_ORIGINAL:167:180:the cam shaft
  label    : ast_eln  score=0.8
  rationale: pitting damage near the cam shaft region indicates elin (elastic limit) failure

  span_key : DOC_OOV:56:64:impeller
  label    : ast_eln  score=0.8
  rationale: Impeller is a type of elongated component

  span_key : DOC_OOV:65:72:surface
  label    : ast_eln  score=0.8
  rationale: oxide layer buildup is characteristic of bearing fatigue failure mode i

---
## Section 7 — End-to-End Integration: NERSeed

`ner_seed_provider_from_pipeline()` wires all methods together into a single callable
that returns a fully-populated `NERSeed` from a chunk dict:

```
chunk text
  ├── HybridNERPipeline  →  systems, components, mechanisms, outcomes, actions, tools, properties
  ├── extract_equipment_ids  →  equipment_ids (+ fallback from full text)
  ├── extract_doc_ref_ids    →  doc_refs
  ├── extract_alarm_ref_ids  →  alarm_ids
  └── SpacyAnnotator         →  measurements, temporal_refs, temporal_relations,
                                temporal_qualifiers, locations, conjectures
```

This is the exact function called at indexing time by `augment_chunks_with_structured_summaries()`.

In [17]:
# ── Build the integrated provider ─────────────────────────────────────────────
from dackar.RCA.summarizers.reliability_summarizer import NERSeed

provider = ner_seed_provider_from_pipeline(
    pipeline   = pipe,
    NERSeed    = NERSeed,
    annotator  = annotator,   # spaCy annotator built in Section 4
)
print("Provider ready.")

Provider ready.


In [18]:
# ── Run provider on a realistic work-order chunk ───────────────────────────────
#
# A single realistic chunk that exercises all NER categories simultaneously.

chunk = {
    "doc_id"   : "WO-87342",
    "chunk_id" : "WO-87342::chunk::3",
    "text"     : (
        "Work order WO-87342 (see CR-2024-0451) was initiated for pump P-101A. "
        "Inspection found adhesive wear on the bearing and evidence of acid attack on the gasket. "
        "Pitting damage was identified near the cam shaft region. "
        "The as-found condition of the seal was degraded due to fretting corrosion. "
        "Bearing wear caused excessive vibration, resulting in coolant leakage. "
        "Alarm ALM-101 had been received approximately 48 hours before the failure. "
        "Coolant flow was 250 gpm; reactor coolant temperature was 285°F at the time. "
        "The pump was possibly operating outside design basis upstream of valve MOV-204. "
        "After bearing replacement the system was returned to as-left acceptable condition."
    ),
}

seed = provider(chunk)

print("=== NERSeed (all fields) ===\n")
for field, val in vars(seed).items():
    if val:
        print(f"  {field:30s}: {val}")

=== NERSeed (all fields) ===

  equipment_ids                 : ['P-101A', 'MOV-204']
  components                    : ['pump P-101A', 'bearing', 'gasket', 'cam', 'shaft', 'Bearing wear', 'excessive vibration', 'coolant leakage', 'Alarm ALM-101', 'Coolant flow', 'temperature', 'design basis', 'valve', 'valve MOV-204']
  mechanisms                    : ['adhesive wear', 'acid attack', 'Pitting', 'fretting corrosion', 'wear', 'vibration']
  outcomes                      : ['attack', 'Pitting damage', 'leakage', 'failure']
  properties                    : ['flow', 'temperature', 'time']
  doc_refs                      : ['CR-2024-0451', 'WO-87342']
  alarm_ids                     : ['ALM-101']


In [19]:
# ── Section 8 — Full summary: Hybrid NER entities across all test texts ────────

rows = []
for res, doc_id in [(res1,'original'), (res2,'oov'), (res3,'events'), (res4,'ambiguous')]:
    for e in res.entities:
        rows.append({
            'doc'   : doc_id,
            'span'  : e.text,
            'labels': ', '.join(e.labels or []),
            'groups': ', '.join(e.groups or []),
            'start' : e.start,
            'end'   : e.end,
        })

pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_rows', 100)
df_ner = pd.DataFrame(rows)
print("Hybrid NER entities:")
display(df_ner)

# ── NERSeed field summary from end-to-end run ──────────────────────────────────
print("\nNERSeed field summary (end-to-end chunk):")
seed_rows = []
for field, val in vars(seed).items():
    seed_rows.append({"field": field, "count": len(val) if isinstance(val, list) else (1 if val else 0), "values": str(val)[:120]})
display(pd.DataFrame(seed_rows))

Hybrid NER entities:


,doc,span,labels,groups,start,end
0,original,adhesive wear,deg_mech,G4_MECHANISM_PROCESS,29,42
1,original,the bearing,ast_eln,G2_PHYSICAL_ASSET_SYSTEM,46,57
2,original,bearing,comp_mech_rot,G1_PHYSICAL_COMPONENT,50,57
3,original,acid attack,deg_mech,G4_MECHANISM_PROCESS,74,85
4,original,attack,event,G5_FAILURE_OUTCOME,79,85
5,original,the gasket,ast_I_C,G2_PHYSICAL_ASSET_SYSTEM,89,99
6,original,gasket,comp_mech_spec,G1_PHYSICAL_COMPONENT,93,99
7,original,degradation,deg_mech,G4_MECHANISM_PROCESS,131,142
8,original,pitting,deg_mech,G4_MECHANISM_PROCESS,147,154
9,original,damage,fail_type_v,G5_FAILURE_OUTCOME,155,161



NERSeed field summary (end-to-end chunk):


,field,count,values
0,systems,0,[]
1,equipment_ids,2,"['P-101A', 'MOV-204']"
2,components,14,"['pump P-101A', 'bearing', 'gasket', 'cam', 's..."
3,mechanisms,6,"['adhesive wear', 'acid attack', 'Pitting', 'f..."
4,outcomes,4,"['attack', 'Pitting damage', 'leakage', 'failu..."
5,surveillance_actions,0,[]
6,maintenance_actions,0,[]
7,properties,3,"['flow', 'temperature', 'time']"
8,tools,0,[]
9,fm_ids,0,[]
